## A. EXTRACT DATA

### Soal

Baca tiga sumber data Kopi Nusantara dari HDFS menggunakan Spark DataFrame, yaitu:

1. Data cabang `tugas7_cabang.json`
2. Data menu `tugas7_menu.csv`
3. Data transaksi `tugas7_transaksi.csv`

Tampilkan jumlah baris masing-masing DataFrame dan schema dari ketiga data tersebut.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Membaca data transaksi
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data cabang
df_cabang = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

# Membaca data menu
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

print("Jumlah transaksi :", df_transaksi.count())
print("Jumlah cabang    :", df_cabang.count())
print("Jumlah menu      :", df_menu.count())

print("\n=== Schema Transaksi ===")
df_transaksi.printSchema()

print("\n=== Schema Cabang ===")
df_cabang.printSchema()

print("\n=== Schema Menu ===")
df_menu.printSchema()

26/10/01 00:16:39 WARN Utils: Your hostname, asih2 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/01 00:16:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 00:16:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Jumlah transaksi : 4000
Jumlah cabang    : 3
Jumlah menu      : 15

=== Schema Transaksi ===
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)


=== Schema Cabang ===
root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)


=== Schema Menu ===
root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



### Penjelasan

Pada tahap Extract, ketiga sumber data dibaca menggunakan Spark DataFrame dari HDFS. Data transaksi terdiri dari 4.000 baris, data cabang terdiri dari 3 baris, dan data menu terdiri dari 15 baris. Schema ditampilkan untuk mengetahui struktur dan tipe data dari masing-masing sumber sebelum dilakukan proses transformasi.

## B. JOIN & TRANSFORM

### Soal

Lakukan join terhadap ketiga DataFrame dengan ketentuan:

1. Data transaksi dihubungkan dengan data cabang berdasarkan `cabang_id`.
2. Data transaksi dihubungkan dengan data menu berdasarkan `menu_id`.
3. Tambahkan kolom `total_penjualan` yang dihitung dari `qty × harga`.
4. Tampilkan hasil penggabungan data.

In [2]:
from pyspark.sql.functions import col

df_gabungan = df_transaksi.join(
    df_cabang,
    on="cabang_id",
    how="inner"
).join(
    df_menu,
    on="menu_id",
    how="inner"
)

df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


### Penjelasan

Pada tahap Transform dilakukan penggabungan data transaksi dengan data cabang dan data menu menggunakan `INNER JOIN`. Penggabungan dilakukan berdasarkan `cabang_id` dan `menu_id`. Setelah data berhasil digabungkan, dibuat kolom `total_penjualan` dengan mengalikan jumlah menu yang terjual (`qty`) dengan harga menu (`harga`).

## C. WINDOW FUNCTION

### Soal

Gunakan window function untuk menentukan **2 menu dengan total penjualan tertinggi pada setiap cabang**.

Gunakan `row_number()` untuk memberikan peringkat berdasarkan `total_penjualan` secara descending pada masing-masing cabang.

In [3]:
from pyspark.sql.functions import sum as spark_sum, row_number
from pyspark.sql.window import Window

# Menghitung total penjualan setiap menu pada setiap cabang
ringkasan_menu = df_gabungan.groupBy(
    "nama_cabang",
    "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Membuat window berdasarkan cabang
window_cabang = Window \
    .partitionBy("nama_cabang") \
    .orderBy(col("total_penjualan").desc())

# Memberikan peringkat menu
top2_menu = ringkasan_menu.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
)

top2_menu.orderBy(
    "nama_cabang",
    "peringkat"
).show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



### Penjelasan

Window function digunakan untuk memberikan peringkat menu pada setiap cabang berdasarkan total penjualan. Fungsi `row_number()` memberikan nomor urut dari penjualan tertinggi ke terendah. Dengan memfilter `peringkat <= 2`, diperoleh dua menu dengan penjualan tertinggi pada masing-masing cabang.

## D. SPARK SQL

### Soal

Gunakan Spark SQL untuk menghitung **total penjualan berdasarkan `kepala_barista`**.

Hasil harus diurutkan dari total penjualan terbesar ke terkecil.

In [4]:
df_gabungan.createOrReplaceTempView("penjualan_kopi")

hasil_sql = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM penjualan_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

hasil_sql.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



### Penjelasan

Data `df_gabungan` terlebih dahulu dibuat menjadi temporary view bernama `penjualan_kopi`. Selanjutnya Spark SQL digunakan untuk menghitung jumlah seluruh penjualan berdasarkan `kepala_barista`. Hasil kemudian diurutkan secara descending sehingga nilai total penjualan terbesar berada di bagian atas.

## E. LOAD KE HDFS

### Soal

Simpan hasil `df_gabungan` ke HDFS dengan ketentuan:

1. Format penyimpanan menggunakan Parquet.
2. Data dipartisi berdasarkan `kategori_menu`.
3. Lokasi penyimpanan adalah `/user/asih2/tugas7/processed/laporan`.
4. Baca kembali data Parquet tersebut.
5. Tampilkan jumlah baris setelah dibaca kembali.
6. Tampilkan schema hasil pembacaan.

In [6]:
hdfs_path = "hdfs://localhost:9000/user/asih2/tugas7/processed/laporan"

# Menyimpan data ke HDFS dalam format Parquet
# dan dipartisi berdasarkan kategori_menu
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(hdfs_path)

print("Data berhasil disimpan ke HDFS:")
print(hdfs_path)

# Membaca kembali data Parquet
df_hasil = spark.read.parquet(hdfs_path)

print("\nJumlah data setelah dibaca kembali:",
      df_hasil.count())

print("\n=== Schema Data Hasil ===")
df_hasil.printSchema()

print("\n=== Contoh Data ===")
df_hasil.show(10)

Data berhasil disimpan ke HDFS:
hdfs://localhost:9000/user/asih2/tugas7/processed/laporan

Jumlah data setelah dibaca kembali: 4000

=== Schema Data Hasil ===
root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- total_penjualan: integer (nullable = true)
 |-- kategori_menu: string (nullable = true)


=== Contoh Data ===
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--

### Penjelasan

Data hasil penggabungan disimpan ke HDFS menggunakan format Parquet dan dipartisi berdasarkan `kategori_menu`. Setelah proses penyimpanan selesai, data dibaca kembali menggunakan `spark.read.parquet()` untuk memastikan data berhasil disimpan. Jumlah data setelah dibaca kembali seharusnya tetap sebanyak 4.000 baris.

In [7]:
!hdfs dfs -ls -R /user/asih2/tugas7/processed/laporan

-rw-r--r--   3 asih2 supergroup          0 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - asih2 supergroup          0 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Kopi
-rw-r--r--   3 asih2 supergroup      12309 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Kopi/part-00000-d5629d91-f200-4b4a-ac12-ade2d11b3f51.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Makanan Ringan
-rw-r--r--   3 asih2 supergroup      16393 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Makanan Ringan/part-00000-d5629d91-f200-4b4a-ac12-ade2d11b3f51.c000.snappy.parquet
drwxr-xr-x   - asih2 supergroup          0 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Non-Kopi
-rw-r--r--   3 asih2 supergroup       8552 2026-10-01 00:21 /user/asih2/tugas7/processed/laporan/kategori_menu=Non-Kopi/part-00000-d5629d91-f200-4b4a-ac12

### Penjelasan

Perintah `hdfs dfs -ls -R` digunakan untuk memeriksa isi direktori hasil penyimpanan pada HDFS. Dari hasil tersebut dapat dilihat direktori partisi berdasarkan kategori menu dan file Parquet yang dihasilkan oleh Spark.

## F. EXECUTIVE SUMMARY & RECOMMENDATION

### Soal

Buat ringkasan eksekutif berdasarkan hasil pengolahan data dengan menampilkan:

1. Total penjualan setiap cabang.
2. Jumlah transaksi setiap cabang.
3. Rata-rata nilai transaksi setiap cabang.

Kemudian buat rekomendasi berdasarkan hasil analisis:
- Cabang mana yang memiliki total penjualan tertinggi?
- Pada cabang dengan penjualan terendah, menu apa yang dapat dipromosikan berdasarkan hasil window function?
- Jelaskan alasannya.

In [9]:
from pyspark.sql.functions import (
    sum as spark_sum,
    count,
    avg
)

ringkasan_eksekutif = df_hasil.groupBy(
    "nama_cabang",
    "kota",
    "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias(
        "total_penjualan"
    ),
    count("trx_id").alias(
        "jumlah_transaksi"
    ),
    avg("total_penjualan").alias(
        "rata_rata_nilai_transaksi"
    )
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



### Penjelasan

Ringkasan eksekutif dibuat dengan melakukan `groupBy` berdasarkan cabang, kota, dan kepala barista. Kemudian dihitung total penjualan, jumlah transaksi, dan rata-rata nilai transaksi pada masing-masing cabang. Hasil diurutkan berdasarkan total penjualan secara descending sehingga cabang dengan total penjualan terbesar berada di bagian atas.

### Rekomendasi

Berdasarkan hasil analisis terhadap 4.000 transaksi Kopi Nusantara, terdapat tiga cabang yang dianalisis, yaitu Magelang, Yogyakarta, dan Semarang. Hasil ringkasan menunjukkan bahwa Kopi Nusantara Semarang memiliki total penjualan sebesar Rp90.542.000 dengan 1.393 transaksi. Kopi Nusantara Magelang memiliki total penjualan sebesar Rp87.138.000 dengan 1.302 transaksi, sedangkan Kopi Nusantara Yogyakarta memiliki total penjualan sebesar Rp84.645.000 dengan 1.305 transaksi. Berdasarkan total penjualan pada dataset ini, cabang Yogyakarta memiliki total penjualan paling rendah.

Berdasarkan hasil window function, dua menu dengan total penjualan tertinggi di cabang Yogyakarta adalah Menu-12 sebesar Rp9.275.000 dan Menu-15 sebesar Rp8.160.000. Kedua menu tersebut dapat dipertimbangkan untuk mendapatkan promosi lebih sering di cabang Yogyakarta karena memiliki kontribusi penjualan yang tinggi pada cabang tersebut. Promosi dapat dilakukan melalui paket menu, penawaran khusus, atau penempatan menu pada area yang mudah dilihat pelanggan. Strategi tersebut menggunakan menu yang sudah menunjukkan penjualan tinggi sebagai salah satu pilihan untuk meningkatkan transaksi pada cabang dengan total penjualan lebih rendah.